In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
from typing import Final

CONNECTION_STRING: Final = f"dbname={os.getenv("DB_NAME")} user={os.getenv("DB_USER")} password={os.getenv("DB_PW")} host={os.getenv("DB_HOST")}"

In [9]:
import psycopg

# Connect to an existing database
with psycopg.connect(f"{CONNECTION_STRING}") as conn:

    # Open a cursor to perform database operations
    with conn.cursor() as cur:

        # Execute a command: this creates a new table
        cur.execute("""
            CREATE TABLE IF NOT EXISTS test (
                id serial PRIMARY KEY,
                num integer,
                data text)
            """)

        # Pass data to fill a query placeholders and let Psycopg perform
        # the correct conversion (no SQL injections!)
        cur.execute(
            "INSERT INTO test (num, data) VALUES (%s, %s)", # %s prevents sql injection
            (100, "abc'def"))

        # Query the database and obtain data as Python objects.
        cur.execute("SELECT * FROM test")
        print(cur.fetchone())
        # will print (1, 100, "abc'def")

        # You can use `cur.executemany()` to perform an operation in batch
        cur.executemany(
            "INSERT INTO test (num) values (%s)",
            [(33,), (66,), (99,)])

        # You can use `cur.fetchmany()`, `cur.fetchall()` to return a list
        # of several records, or even iterate on the cursor
        cur.execute("SELECT id, num FROM test order by num")
        for record in cur:
            print(record)

        # Make the changes to the database persistent
        conn.commit()

(1, 100, "abc'def")
(2, 33)
(6, 33)
(7, 66)
(3, 66)
(8, 99)
(4, 99)
(5, 100)
(1, 100)


In [15]:
import psycopg

with psycopg.connect(f"{CONNECTION_STRING}") as connector:
    with connector.cursor() as cursor:

        # execute 1 single command
        cursor.execute(
            """INSERT INTO has_parking_spot (centre_id, wing, floor, spot_number, availability)
                    VALUES (%s,%s, %s,%s,%s)""",
                    (3,"Seasons Mall Parking Centre Wing 1","B1","A-01",True))
        
        # execute many commands
        cursor.executemany( """INSERT INTO has_parking_spot (centre_id, wing, floor, spot_number, availability)
                                 VALUES (%s,%s, %s,%s,%s)""",
                                [   (3,"Seasons Mall Parking Centre Wing 1","B1","A-02",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B1","A-03",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B1","A-04",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B1","B-01",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B1","B-03",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B1","B-04",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B2","A-01",True),
                                    (3,"Seasons Mall Parking Centre Wing 1","B2","A-02",True),
                                 ])

UniqueViolation: duplicate key value violates unique constraint "unique_floor"
DETAIL:  Key (centre_id, wing, floor)=(3, Seasons Mall Parking Centre Wing 1, B1) already exists.